In [16]:
!pip install pandas openpyxl python-docx seaborn matplotlib

In [17]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from docx import Document
from docx.shared import Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH

In [18]:
file_name = "Delinquency_prediction_dataset (1).xlsx"

df = pd.read_excel(file_name)

# Basic info
num_records = df.shape[0]
num_columns = df.shape[1]

df.head()

,Customer_ID,Age,Income,Credit_Score,Credit_Utilization,Missed_Payments,Delinquent_Account,Loan_Balance,Debt_to_Income_Ratio,Employment_Status,Account_Tenure,Credit_Card_Type,Location,Month_1,Month_2,Month_3,Month_4,Month_5,Month_6
0,CUST0001,56,165580.0,398.0,0.390502,3,0,16310.0,0.317396,EMP,18,Student,Los Angeles,Late,Late,Missed,Late,Missed,Late
1,CUST0002,69,100999.0,493.0,0.312444,6,1,17401.0,0.196093,Self-employed,0,Standard,Phoenix,Missed,Missed,Late,Missed,On-time,On-time
2,CUST0003,46,188416.0,500.0,0.359930,0,0,13761.0,0.301655,Self-employed,1,Platinum,Chicago,Missed,Late,Late,On-time,Missed,Late
3,CUST0004,32,101672.0,413.0,0.371400,3,0,88778.0,0.264794,Unemployed,15,Platinum,Phoenix,Late,Missed,Late,Missed,Late,Late
4,CUST0005,60,38524.0,487.0,0.234716,2,0,13316.0,0.510583,Self-employed,11,Standard,Phoenix,Missed,On-time,Missed,Late,Late,Late


In [19]:
# Data types
data_types = df.dtypes.astype(str)

# Numerical and categorical columns
num_cols = df.select_dtypes(include=np.number).columns.tolist()
cat_cols = df.select_dtypes(exclude=np.number).columns.tolist()

# Duplicate rows
duplicates = df.duplicated().sum()

overview = {
    "records": num_records,
    "columns": num_columns,
    "numerical_columns": num_cols,
    "categorical_columns": cat_cols,
    "duplicates": duplicates
}

overview

{'records': 500,
 'columns': 19,
 'numerical_columns': ['Age',
  'Income',
  'Credit_Score',
  'Credit_Utilization',
  'Missed_Payments',
  'Delinquent_Account',
  'Loan_Balance',
  'Debt_to_Income_Ratio',
  'Account_Tenure'],
 'categorical_columns': ['Customer_ID',
  'Employment_Status',
  'Credit_Card_Type',
  'Location',
  'Month_1',
  'Month_2',
  'Month_3',
  'Month_4',
  'Month_5',
  'Month_6'],
 'duplicates': np.int64(0)}

In [20]:
missing = df.isnull().sum()

missing_percent = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    "Missing Count": missing,
    "Missing Percent": missing_percent
})

missing_df = missing_df[missing_df["Missing Count"] > 0]

missing_df

,Missing Count,Missing Percent
Income,39,7.8
Credit_Score,2,0.4
Loan_Balance,29,5.8


In [21]:
stats_summary = df.describe().T

stats_summary

,count,mean,std,min,25%,50%,75%,max
Age,500.0,46.266000,16.187629,18.00,33.000000,46.500000,59.250000,74.000000
Income,461.0,108379.893709,53662.723741,15404.00,62295.000000,107658.000000,155734.000000,199943.000000
Credit_Score,498.0,577.716867,168.881211,301.00,418.250000,586.000000,727.250000,847.000000
Credit_Utilization,500.0,0.491446,0.197103,0.05,0.356486,0.485636,0.634440,1.025843
Missed_Payments,500.0,2.968000,1.946935,0.00,1.000000,3.000000,5.000000,6.000000
Delinquent_Account,500.0,0.160000,0.366973,0.00,0.000000,0.000000,0.000000,1.000000
Loan_Balance,471.0,48654.428875,29395.537273,612.00,23716.500000,45776.000000,75546.500000,99620.000000
Debt_to_Income_Ratio,500.0,0.298862,0.094521,0.10,0.233639,0.301634,0.362737,0.552956
Account_Tenure,500.0,9.740000,5.923054,0.00,5.000000,10.000000,15.000000,19.000000


In [22]:
correlation_matrix = df.corr(numeric_only=True)

# Find strongest correlations
corr_pairs = correlation_matrix.unstack()

corr_pairs = corr_pairs.sort_values(ascending=False)

# Remove duplicate/self correlation
corr_pairs = corr_pairs[corr_pairs != 1]

top_corr = corr_pairs.head(10)

top_corr

,,0
Credit_Score,Income,0.071287
Income,Credit_Score,0.071287
Account_Tenure,Credit_Utilization,0.065264
Credit_Utilization,Account_Tenure,0.065264
Loan_Balance,Debt_to_Income_Ratio,0.056971
Debt_to_Income_Ratio,Loan_Balance,0.056971
Account_Tenure,Loan_Balance,0.054607
Loan_Balance,Account_Tenure,0.054607
Delinquent_Account,Income,0.045409
Income,Delinquent_Account,0.045409


In [23]:
anomalies = {}

for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower) | (df[col] > upper)]

    anomalies[col] = len(outliers)

anomalies

{'Age': 0,
 'Income': 0,
 'Credit_Score': 0,
 'Credit_Utilization': 0,
 'Missed_Payments': 0,
 'Delinquent_Account': 80,
 'Loan_Balance': 0,
 'Debt_to_Income_Ratio': 0,
 'Account_Tenure': 0}

In [24]:
# Load template
doc = Document("EDA_SummaryReport_Template.docx")

# Add content

doc.add_heading("EDA Summary Report", level=0)


# 1. Introduction
doc.add_heading("1. Introduction", level=1)

doc.add_paragraph(
"This report presents an Exploratory Data Analysis (EDA) of the delinquency prediction dataset. "
"The goal is to understand dataset structure, identify missing values, detect anomalies, "
"and discover key risk indicators that may impact delinquency prediction."
)


# 2. Dataset Overview
doc.add_heading("2. Dataset Overview", level=1)

doc.add_paragraph(f"Number of records: {num_records}")
doc.add_paragraph(f"Number of columns: {num_columns}")

doc.add_paragraph(f"Numerical variables: {', '.join(num_cols)}")
doc.add_paragraph(f"Categorical variables: {', '.join(cat_cols)}")

doc.add_paragraph(f"Duplicate rows found: {duplicates}")


# 3. Missing Data Analysis
doc.add_heading("3. Missing Data Analysis", level=1)

if len(missing_df) == 0:
    doc.add_paragraph("No missing values detected in dataset.")
else:
    for col in missing_df.index:
        doc.add_paragraph(
            f"{col}: {missing_df.loc[col,'Missing Count']} missing values "
            f"({missing_df.loc[col,'Missing Percent']:.2f}%)"
        )


# 4. Key Findings and Risk Indicators
doc.add_heading("4. Key Findings and Risk Indicators", level=1)

doc.add_paragraph("Top correlations observed:")

for index, value in top_corr.items():
    doc.add_paragraph(f"{index}: {value:.3f}")

doc.add_paragraph("Anomaly counts per column:")

for col, count in anomalies.items():
    doc.add_paragraph(f"{col}: {count} anomalies")


# 5. AI & GenAI Usage
doc.add_heading("5. AI & GenAI Usage", level=1)

doc.add_paragraph(
"AI tools were used to automate EDA, identify missing values, correlations, "
"and anomalies. Python libraries such as Pandas, NumPy, and Seaborn were used."
)


# 6. Conclusion
doc.add_heading("6. Conclusion & Next Steps", level=1)

doc.add_paragraph(
"The dataset has been successfully analyzed. Missing values, anomalies, and correlations "
"have been identified. Next steps include data preprocessing, feature engineering, "
"and model development for delinquency prediction."
)


# Save report
report_name = "EDA_Summary_Report_Output.docx"
doc.save(report_name)

print("Report Generated Successfully!")

Report Generated Successfully!


In [25]:
files.download("EDA_Summary_Report_Output.docx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>